In [1]:
!pip install -q torchinfo
!if [ ! -d EcoSort-AI ]; then git clone https://github.com/DinoKiller0607/EcoSort-AI.git; fi
import sys
sys.path.append('/kaggle/working/EcoSort-AI')

Cloning into 'EcoSort-AI'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 46 (delta 20), reused 34 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 571.38 KiB | 3.83 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
import torch, torchvision
from torch import nn
from torchinfo import summary
import torchvision.transforms.v2 as transforms
import data_setup, engine
from utils import save_model, plot_loss_curves

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.__version__)

cuda 2.11.0+cu128


In [3]:
from pathlib import Path
data_path = Path('/kaggle/input/datasets/dinokiller607/waste-dataset-dry-wet-recyclable-ewaste/data')   # paste the copied path here
train_dir, val_dir, test_dir = data_path/'train', data_path/'val', data_path/'test'
!ls {data_path}

test  train  val


In [4]:
weights = torchvision.models.ViT_L_16_Weights.DEFAULT

auto_transforms = weights.transforms()

train_dataloader, val_dataloader, class_names = data_setup.create_weighted_dataloaders(
    train_dir=train_dir,
    val_dir=val_dir,
    transform=auto_transforms,
    batch_size=64,
    num_workers=2
)
class_names

['dry', 'ewaste', 'recyclable', 'wet']

In [5]:
# Check whether the classes are balanced
from collections import Counter

def label_counts(loader, max_batches=100):
    c = Counter()
    for i, (_, y) in enumerate(loader):
        if i >= max_batches: break
        c.update(y.tolist())
    return {class_names[k]: v for k, v in sorted(c.items())}

print('train (first 100 batches):', label_counts(train_dataloader))
print('val   (first 100 batches):', label_counts(val_dataloader))

train (first 100 batches): {'dry': 1623, 'ewaste': 1566, 'recyclable': 1593, 'wet': 1618}
val   (first 100 batches): {'dry': 1623, 'ewaste': 1566, 'recyclable': 1593, 'wet': 1618}


In [6]:
# Create another instance of our pretrained model and feed it the weighted dataloaders.
model_3 = torchvision.models.vit_l_16(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/vit_l_16-852ce7e3.pth" to /root/.cache/torch/hub/checkpoints/vit_l_16-852ce7e3.pth


100%|██████████| 1.13G/1.13G [00:05<00:00, 217MB/s]


In [7]:
# Print with torchinfo
from torchinfo import summary
summary(model=model_3,
        input_size=(1, 3, 224, 224),    # an example of [batch size, color_channels, height, width]
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
VisionTransformer (VisionTransformer)                        [1, 3, 224, 224]     [1, 1000]            1,024                True
├─Conv2d (conv_proj)                                         [1, 3, 224, 224]     [1, 1024, 14, 14]    787,456              True
├─Encoder (encoder)                                          [1, 197, 1024]       [1, 197, 1024]       201,728              True
│    └─Dropout (dropout)                                     [1, 197, 1024]       [1, 197, 1024]       --                   --
│    └─Sequential (layers)                                   [1, 197, 1024]       [1, 197, 1024]       --                   True
│    │    └─EncoderBlock (encoder_layer_0)                   [1, 197, 1024]       [1, 197, 1024]       12,596,224           True
│    │    └─EncoderBlock (encoder_layer_1)                   [1, 197, 1024]       [1, 197, 102

In [8]:
for param in model_3.parameters():
    param.requires_grad = False
for param in model_3.encoder.ln.parameters():
    param.requires_grad = True

In [9]:
# Update the classifier head to suit our problems
model_3.heads = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1024, # feature vector coming in
              out_features=len(class_names)).to(device))    # how many classes do we have?
model_3.heads

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1024, out_features=4, bias=True)
)

In [10]:
model_3 = model_3.to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model_3 = nn.DataParallel(model_3)

Using 2 GPUs


In [11]:
summary(model=model_3,
        input_size=(1, 3, 224, 224),    # an example of [batch size, color_channels, height, width]
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                           Input Shape          Output Shape         Param #              Trainable
DataParallel (DataParallel)                                       [1, 3, 224, 224]     [1, 4]               --                   Partial
├─VisionTransformer (module)                                      [1, 3, 224, 224]     [1, 4]               303,305,732          Partial
├─VisionTransformer (module)                                      --                   --                   --                   Partial
│    └─Conv2d (conv_proj)                                         [1, 3, 224, 224]     [1, 1024, 14, 14]    (787,456)            False
│    └─Encoder (encoder)                                          [1, 197, 1024]       [1, 197, 1024]       302,513,152          Partial
│    └─Encoder (encoder)                                          --                   --                   --                   Partial
│    │    └─Dropout (dropout)            

In [12]:
NUM_EPOCHS = 15      # set to 1 for the test run in Step 11

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_3.parameters(), lr=1e-4)   # your model/optimizer names

from timeit import default_timer as timer
start_time = timer()

results = engine.train(model=model_3, train_dataloader=train_dataloader,
                       val_dataloader=val_dataloader, optimizer=optimizer,
                       loss_fn=loss_fn, epochs=NUM_EPOCHS, device=device)

save_model(model_3.module if isinstance(model_3, nn.DataParallel) else model_3,
             target_dir='/kaggle/working/models', model_name='04_pretrained_transformer.pt')

import json
json.dump(results, open('/kaggle/working/transformer_results.json', 'w'))

print(f"[INFO] Total training time: {timer()-start_time:.3f} seconds")

  0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.7542 | train_acc: 0.7715 | train_f1: 0.7710 | val_loss: 0.4524 | val_acc: 0.8966 | val_f1: 0.8967
Epoch: 2 | train_loss: 0.3750 | train_acc: 0.9007 | train_f1: 0.9002 | val_loss: 0.3172 | val_acc: 0.9138 | val_f1: 0.9140
Epoch: 3 | train_loss: 0.2934 | train_acc: 0.9159 | train_f1: 0.9161 | val_loss: 0.2648 | val_acc: 0.9197 | val_f1: 0.9199
Epoch: 4 | train_loss: 0.2566 | train_acc: 0.9248 | train_f1: 0.9246 | val_loss: 0.2347 | val_acc: 0.9273 | val_f1: 0.9273
Epoch: 5 | train_loss: 0.2320 | train_acc: 0.9306 | train_f1: 0.9300 | val_loss: 0.2169 | val_acc: 0.9333 | val_f1: 0.9333
Epoch: 6 | train_loss: 0.2116 | train_acc: 0.9359 | train_f1: 0.9356 | val_loss: 0.2048 | val_acc: 0.9379 | val_f1: 0.9379
Epoch: 7 | train_loss: 0.2048 | train_acc: 0.9385 | train_f1: 0.9383 | val_loss: 0.1941 | val_acc: 0.9385 | val_f1: 0.9386
Epoch: 8 | train_loss: 0.1931 | train_acc: 0.9406 | train_f1: 0.9403 | val_loss: 0.1872 | val_acc: 0.9400 | val_f1: 0.9401
Epoch: 9 | train

In [13]:
results

{'train_loss': [0.7542300034759096,
  0.3750316848843532,
  0.293434087602593,
  0.2565711439530677,
  0.23199906891281125,
  0.21157671678847761,
  0.20475345752302215,
  0.19305802741745914,
  0.1850039865639345,
  0.17663808199057607,
  0.17813751062126776,
  0.17680192222625543,
  0.17095595602594943,
  0.16404249583992472,
  0.1609051931000619],
 'train_acc': [np.float64(0.7714574217435207),
  np.float64(0.9007068327162572),
  np.float64(0.9158838468835103),
  np.float64(0.9248187019981029),
  np.float64(0.9306018787674796),
  np.float64(0.9359260732535725),
  np.float64(0.9385269728588477),
  np.float64(0.9405770937241823),
  np.float64(0.9443407484471099),
  np.float64(0.9448303295492794),
  np.float64(0.9460848811235887),
  np.float64(0.945534102383648),
  np.float64(0.9477678161622961),
  np.float64(0.9490835653743765),
  np.float64(0.9510418897830544)],
 'train_precision': [0.7727377029683375,
  0.9003265077518493,
  0.9164468707553921,
  0.9248834701303787,
  0.9305370130604